Welcome to the study notes for **Introduction to Large Language Models (LLMs)**. These notes are specifically tailored for your NPTEL exam preparation, covering the lecture on **Sequence-to-Sequence (Seq2Seq) Modeling** using Recurrent Neural Networks (RNNs).

---

## 1. Introduction to Sequence-to-Sequence (Seq2Seq) Models

A **Seq2Seq model** is designed to take a sequence of data as input and generate a new sequence as output. These models became prominent around 2015-2016 and serve as the foundation for modern language modeling.

### Problem Mappings

Seq2Seq architectures can handle different input-output dimensionalities:

* **Many-to-Many:** Both input and output are sequences.
* *Examples:* Machine Translation (French to English), Text Summarization (long document to short document), Dialogue generation.


* **One-to-Many:** The input is a single item, and the output is a sequence.
* *Examples:* Image Captioning (Input: Image, Output: Text sequence), Speech/Music generation from a single prompt.
* *Note:* In image captioning, an image matrix can technically be flattened into a sequence, making it many-to-many under the hood, but at a high conceptual level, it acts as one-to-many.



---

## 2. Seq2Seq Architecture: The Encoder-Decoder Network

Because there are two distinct sequences involved (input and output), a standard Seq2Seq model utilizes **two separate RNNs**.

### The Encoder

The Encoder processes the input sequence (e.g., a French sentence) token by token and compresses it into a final hidden state.

* **Weight Matrices Present:**
* $W_e$: The embedding weight matrix.
* $W_h$: The hidden-to-hidden transition matrix.


* **Weight Matrices Absent:**
* $U$: The linear projection matrix is **not present** in the encoder. This is because the encoder does not predict or output any vocabulary tokens; its only job is to encode information.



### The Decoder

The Decoder takes the final hidden state of the Encoder as its initial state and generates the output sequence (e.g., an English sentence) token by token.

* **Weight Matrices Present:**
* $W'_e$: The embedding matrix for the target language.
* $W'_h$: The hidden-to-hidden transition matrix.
* $U$: The linear projection matrix. This projects the decoder's hidden state into a $V$-dimensional vector (where $V$ is the vocabulary size, e.g., 1 million tokens). A softmax layer then converts this into a probability distribution over the vocabulary.



> **NPTEL Exam Tip:**
> * If the input and output languages are different (e.g., Machine Translation), the embedding matrices ($W_e$ and $W'_e$) are completely different.
> * If the input and output languages are the same (e.g., Text Summarization), the embedding weights can be shared.
> 
> 

---

## 3. Training the Seq2Seq Model

Training an encoder-decoder network involves calculating the loss at every generated step and updating the entire network simultaneously.

### Teacher Forcing

During training, the model attempts to predict the next word based on its previous predictions. If it generates a wrong word early on (e.g., generating "she" instead of "he"), all subsequent predictions will be completely skewed, making learning unstable.

**Teacher Forcing** is a training technique where you feed the **actual ground truth token** from the dataset into the next RNN state, regardless of what the model actually predicted.

* *Why it matters:* It acts as "cheating" to guide the model back on track, making training significantly faster and more effective.
* *Limitation:* Teacher forcing can **only** be used during training. During inference (when generating new text), the ground truth is unknown, so the model must rely on its own previous outputs.

### Loss Computation and Backpropagation

* The error/loss ($J_t$) is computed at the hidden state output of each timestamp in the decoder.
* The total error is the sum of errors at all timestamps, normalized by the number of output tokens $T$:

$$J = \frac{1}{T} \sum_{t=1}^{T} J_t$$


* This loss is backpropagated through the decoder and all the way back through the encoder using **Backpropagation Through Time (BPTT)**. Because both RNNs are updated simultaneously, this is called **end-to-end training**.

---

## 4. Seq2Seq as a Conditional Language Model

A standard language model predicts the next word based on previous words. A Seq2Seq model is fundamentally a **Conditional Language Model** because its predictions are conditioned not just on previously generated tokens, but also on the entire input sequence $X$.

Let $X$ be the input sequence and $y_1, y_2, \dots, y_T$ be the output sequence. The probabilities are computed as:

1. **First token:** $P(y_1 \vert{} X)$ (Conditioned solely on the encoder's final hidden state).
2. **Second token:** $P(y_2 \vert{} y_1, X)$
3. **$t$-th token:** $P(y_t \vert{} y_1, y_2, \dots, y_{t-1}, X)$

---

## 5. Inference and Decoding Strategies

Once the model is trained, we need a strategy to select the best sequence of words during the inference (testing) stage.

### 1. Greedy Decoding

At each timestamp, simply pick the single token that has the highest probability.

* **The Problem:** You cannot backtrack. If the model makes a suboptimal choice early on (e.g., translating "He hit me" as "He hit a..."), it is permanently locked into that path.

### 2. Exhaustive Search

To find the absolute best sequence, you could evaluate every possible token combination. At each step, generate all $V$ (vocabulary size) branches.

* **The Problem:** The complexity is $V^T$ where $T$ is the number of timestamps. If $V = 1,000,000$, this is computationally impossible.

### 3. Beam Search (The Middle Ground)

Beam search is a constrained heuristic search. Instead of keeping just 1 hypothesis (greedy) or $V$ hypotheses (exhaustive), you keep track of the **top $K$ hypotheses**, where $K$ is the "beam size" (e.g., $K=2$, $5$, or $10$).

* **Mechanism:**
1. Start with the special `START` token.
2. Generate probabilities for the next tokens and pick the top $K$ choices.
3. From each of those $K$ choices, generate the next set of tokens.
4. Calculate the cumulative score for all branches and prune the tree, keeping only the top $K$ overall paths.
*(Note: While the lecture transcript casually mentions the complexity scales as $K^T$, keeping strictly to the top $K$ candidates at each pruning step practically bounds the exponential explosion, making it highly efficient compared to $V^T$.)*


* **Scoring Hypotheses:**
The score of a branch is the sum of the **log probabilities** of its tokens. Since probabilities are decimals between 0 and 1, their logs are negative numbers. We choose the hypothesis with the **maximum score** (the one closest to 0, or "less negative").

$$\text{Score} = \sum_{t=1}^{T} \log P(y_t \vert{} y_{<t}, X)$$


* **Stopping Criterion:** A branch stops expanding when it generates a special token like `EOD` (End of Decoding) or a full stop.
* **Length Penalty Problem:** Because we are adding negative numbers, a longer sequence will inherently have a more negative (worse) score simply because it has more tokens. This unfairly penalizes long, correct sentences in favor of short, incorrect ones.
* **The Solution:** Divide the final score by the number of tokens generated ($T$) to calculate a **normalized score**.

---

## 6. NPTEL Practice Questions

**Q1: In an Encoder-Decoder Seq2Seq architecture for Machine Translation, which of the following weight matrices is absent in the Encoder?**
A) Embedding Matrix ($W_e$)
B) Hidden-to-Hidden Matrix ($W_h$)
C) Linear Projection Matrix ($U$)
D) None of the above
**Answer: C.** The linear projection matrix $U$ projects hidden states to the vocabulary size to predict words. The encoder does not generate words; it only compresses the input, so $U$ is not needed.

**Q2: What is "Teacher Forcing" in the context of sequence generation?**
A) Forcing the model to output a specific translation regardless of the input.
B) Feeding the model's own predicted output from timestamp $t-1$ as input to timestamp $t$.
C) Feeding the actual ground truth token from timestamp $t-1$ as input to timestamp $t$ during training.
D) Normalizing the loss function by the total number of timestamps.
**Answer: C.** Teacher forcing "cheats" by giving the model the correct previous word during training to prevent cascading errors.

**Q3: A developer implements Beam Search with a beam size of $K=5$. After generating several tokens, they notice the model strongly prefers generating very short, truncated sentences over detailed, correct ones. What is the most likely cause?**
A) The model was not trained with teacher forcing.
B) The developer failed to normalize the log-probability scores by the sequence length.
C) The beam size $K$ is too large.
D) Greedy decoding is accidentally turned on.
**Answer: B.** Log probabilities are negative. Summing more of them for longer sequences results in lower overall scores. Normalizing by the sequence length fixes this bias.

**Q4: Which equation correctly represents the loss function computed over $T$ target tokens in a Seq2Seq model?**
A) $J = \sum_{t=1}^{T} J_t$
B) $J = \frac{1}{T} \sum_{t=1}^{T} J_t$
C) $J = \prod_{t=1}^{T} J_t$
D) $J = \max(J_t)$
**Answer: B.** The total error is the sum of errors at each timestamp normalized (averaged) by the number of output tokens $T$.

---

## 7. Quick Revision Summary

* **Seq2Seq** utilizes two RNNs: an **Encoder** (processes input, has no projection matrix) and a **Decoder** (generates output, projects hidden states to vocab space).
* It operates as a **Conditional Language Model**, determining target token probabilities based on previously generated tokens *and* the entire input sequence.
* **Teacher Forcing** accelerates training by passing ground-truth previous tokens to the current decoder step.
* **End-to-End Training:** Total sequence loss is backpropagated through both the Decoder and Encoder (BPTT).
* **Decoding Strategies:**
* *Greedy:* Picks top token at each step. Fast, but cannot recover from mistakes.
* *Exhaustive:* Explores all combinations ($V^T$). Impossible to compute.
* *Beam Search:* Explores top $K$ paths. Scores using the sum of log probabilities. **Must normalize** scores by length to avoid unfairly penalizing long sentences.